In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-itiawy17/unsloth_089a4a82eca240e08dcd5e3a8ba71c1e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-itiawy17/unsloth_089a4a82eca240e08dcd5e3a8ba71c1e
  Resolved https://github.com/unslothai/unsloth.git to commit d1f9ab659fc5d3309e9e40166be309369bedf852
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6


In [ ]:
# Gọi Unsloth đầu tiên để tối ưu tốc độ
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. Tải não bộ AI (Bản 1.5 Tỷ tham số)
print("Tải mô hình")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. Cấu hình kỹ thuật LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Nạp và chuẩn bị dữ liệu
print(" Nạp file TrainTKB.jsonl")
dataset = load_dataset("json", data_files="TrainTKB.jsonl", split="train")

def format_prompt(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompt, batched=True)

# 4. Huấn luyện (Dùng chuẩn MỚI NHẤT của TRL)
print("Khởi tạo cấu hình SFTConfig")
training_args = SFTConfig(
    output_dir = "tkb_outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    seed = 3407,
    dataset_text_field = "text",
    max_length = 2048,
    dataset_num_proc = 2,
    packing = False,
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

print(" Bắt đầu huấn luyện trên GPU")
trainer.train()

# 5. Lưu kết quả
print("Hoàn tất huấn luyện. Đang lưu mô hình")
model.save_pretrained("tkb_model_lora")
tokenizer.save_pretrained("tkb_model_lora")
print("Thành công! AI đã học xong khóa TKB.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Đang tải mô hình...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


 Đang nạp file TrainTKB.jsonl...
Khởi tạo cấu hình SFTConfig...


...

Step,Training Loss
10,2.053340
20,0.674537
30,0.401558
40,0.280887
50,0.212044
60,0.179729


Unsloth: Restored added_tokens_decoder metadata in tkb_outputs/checkpoint-60/tokenizer_config.json.


Đã xong! Đang lưu mô hình...


Unsloth: Restored added_tokens_decoder metadata in tkb_model_lora/tokenizer_config.json.


THÀNH CÔNG! AI đã học xong khóa TKB.


In [ ]:
# Đóng gói thành định dạng GGUF (Nén 4-bit cực nhẹ cho iPhone)
print("Tiến hành nén mô hình sang chuẩn GGUF cho iOS")
model.save_pretrained_gguf("Model_TKB_iOS", tokenizer, quantization_method = "q4_k_m")
print(" Đóng gói hoàn tất.")

Đang tiến hành nén mô hình sang chuẩn GGUF cho iOS...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Model_TKB_iOS/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:11<00:00, 71.58s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:17<00:00, 77.38s/it]


Unsloth: Merge process complete. Saved to `/content/Model_TKB_iOS`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Model_TKB_iOS_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Model_TKB_iOS_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model Model_TKB_iOS_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to Model_TKB_iOS_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f Model_TKB_iOS_gguf/Modelfile
 Đóng gói hoàn tất! Hãy tải file về máy.
